# direction prediction classifier (binary, optimized)

In [1]:
import pandas as pd
import numpy as np
import xgboost as xgb
from sklearn.metrics import classification_report, accuracy_score, confusion_matrix
from sklearn.preprocessing import StandardScaler
import talib


In [2]:
# Load aat.us.csv data
data = pd.read_csv('../data/aat.us.csv')
data.columns = [c.lower() for c in data.columns]
data['date'] = pd.to_datetime(data['date'])
data = data.set_index('date')
data = data.tail(1000)  # Use last 1000 rows for consistency


In [3]:
def create_features(df):
    """
    Create comprehensive features from OHLCV data
    """
    df = df.copy()

    # Price-based features
    df['return_1'] = df['close'].pct_change(1)
    df['return_3'] = df['close'].pct_change(3)
    df['return_5'] = df['close'].pct_change(5)
    df['return_10'] = df['close'].pct_change(10)
    df['return_20'] = df['close'].pct_change(20)

    # Moving averages
    df['sma_5'] = df['close'].rolling(5).mean()
    df['sma_10'] = df['close'].rolling(10).mean()
    df['sma_20'] = df['close'].rolling(20).mean()
    df['sma_50'] = df['close'].rolling(50).mean()

    # Distance from moving averages
    df['dist_sma5'] = (df['close'] - df['sma_5']) / df['sma_5']
    df['dist_sma20'] = (df['close'] - df['sma_20']) / df['sma_20']
    df['dist_sma50'] = (df['close'] - df['sma_50']) / df['sma_50']

    # Momentum indicators
    df['rsi_14'] = talib.RSI(df['close'], timeperiod=14)
    df['rsi_21'] = talib.RSI(df['close'], timeperiod=21)

    # MACD
    df['macd'], df['macd_signal'], df['macd_hist'] = talib.MACD(
        df['close'], fastperiod=12, slowperiod=26, signalperiod=9
    )

    # Bollinger Bands
    df['bb_upper'], df['bb_middle'], df['bb_lower'] = talib.BBANDS(
        df['close'], timeperiod=20
    )
    df['bb_position'] = (df['close'] - df['bb_lower']) / (df['bb_upper'] - df['bb_lower'])
    df['bb_width'] = (df['bb_upper'] - df['bb_lower']) / df['bb_middle']

    # Volatility
    df['atr_14'] = talib.ATR(df['high'], df['low'], df['close'], timeperiod=14)
    df['atr_20'] = talib.ATR(df['high'], df['low'], df['close'], timeperiod=20)
    df['volatility'] = df['close'].rolling(20).std()

    # Volume indicators
    df['volume_sma'] = df['volume'].rolling(20).mean()
    df['volume_ratio'] = df['volume'] / df['volume_sma']
    df['volume_change'] = df['volume'].pct_change(1)

    # Price patterns
    df['high_low_ratio'] = df['high'] / df['low']
    df['body_size'] = abs(df['close'] - df['open']) / df['open']
    df['upper_shadow'] = (df['high'] - df[['open', 'close']].max(axis=1)) / df['open']
    df['lower_shadow'] = (df[['open', 'close']].min(axis=1) - df['low']) / df['open']

    # Trend strength
    df['adx'] = talib.ADX(df['high'], df['low'], df['close'], timeperiod=14)

    # Stochastic
    df['stoch_k'], df['stoch_d'] = talib.STOCH(
        df['high'], df['low'], df['close'],
        fastk_period=14, slowk_period=3, slowd_period=3
    )

    # Rate of change
    df['roc_5'] = talib.ROC(df['close'], timeperiod=5)
    df['roc_10'] = talib.ROC(df['close'], timeperiod=10)

    return df


In [4]:
# ============================================
# 2. TARGET CREATION (BINARY)
# ============================================

def create_target_binary(df, horizon=1, threshold=0.002):
    """
    Create binary direction labels (Up=1, Down/Neutral=0)

    horizon: how many periods ahead to predict
    threshold: % change needed to be considered up (0.002 = 0.2%)

    Returns:
    1 = Up
    0 = Down/Neutral
    """
    df = df.copy()

    # Calculate future return
    df['future_return'] = df['close'].shift(-horizon) / df['close'] - 1

    # Create labels
    df['target'] = np.where(df['future_return'] > threshold, 1, 0)  # Up = 1, Down/Neutral = 0

    return df


In [5]:
# ============================================
# 3. TRAINING PIPELINE (BINARY)
# ============================================

def train_xgboost_direction_binary(df, test_size=0.2, horizon=1, threshold=0.002):
    """
    Training pipeline for XGBoost binary direction prediction
    """

    # Feature engineering
    print("Creating features...")
    df = create_features(df)
    df = create_target_binary(df, horizon=horizon, threshold=threshold)

    # Remove NaN values
    df = df.dropna()

    # Select features (exclude OHLCV and intermediate calculations)
    feature_cols = [
        'return_1', 'return_3', 'return_5', 'return_10', 'return_20',
        'dist_sma5', 'dist_sma20', 'dist_sma50',
        'rsi_14', 'rsi_21',
        'macd', 'macd_signal', 'macd_hist',
        'bb_position', 'bb_width',
        'atr_14', 'atr_20', 'volatility',
        'volume_ratio', 'volume_change',
        'body_size', 'upper_shadow', 'lower_shadow',
        'adx', 'stoch_k', 'stoch_d',
        'roc_5', 'roc_10'
    ]

    X = df[feature_cols]
    y = df['target']

    # Time-based split (NO SHUFFLING!)
    split_idx = int(len(X) * (1 - test_size))
    X_train, X_test = X[:split_idx], X[split_idx:]
    y_train, y_test = y[:split_idx], y[split_idx:]

    print(f"Training samples: {len(X_train)}")
    print(f"Test samples: {len(X_test)}")
    print(f"Class distribution (train): {np.bincount(y_train)}")

    # Early stopping and reduced complexity
    model = xgb.XGBClassifier(
        n_estimators=100,
        max_depth=3,
        learning_rate=0.05,
        subsample=0.8,
        colsample_bytree=0.8,
        gamma=0.1,
        min_child_weight=3,
        objective='binary:logistic',
        random_state=42,
        n_jobs=-1,
        eval_metric='logloss'  # <-- pass here, not to fit()
        # use_label_encoder removed for recent xgboost
    )

    # Early stopping
    model.fit(
        X_train, y_train,
        eval_set=[(X_test, y_test)],
        verbose=20,
    )

    y_pred = model.predict(X_test)
    y_pred_proba = model.predict_proba(X_test)[:, 1]

    print("\n" + "="*50)
    print("EVALUATION RESULTS (BINARY)")
    print("="*50)

    accuracy = accuracy_score(y_test, y_pred)
    print(f"\nOverall Accuracy: {accuracy:.4f}")

    print("\nClassification Report:")
    print(classification_report(
        y_test, y_pred,
        target_names=['Down/Neutral (0)', 'Up (1)']
    ))

    print("Confusion Matrix:")
    print(confusion_matrix(y_test, y_pred))

    # Feature importance
    print("\nTop 10 Most Important Features:")
    feature_importance = pd.DataFrame({
        'feature': feature_cols,
        'importance': model.feature_importances_
    }).sort_values('importance', ascending=False)
    print(feature_importance.head(10).to_string(index=False))

    # High-confidence predictions only
    high_conf_mask = y_pred_proba > 0.6
    if high_conf_mask.sum() > 0:
        high_conf_accuracy = accuracy_score(
            y_test[high_conf_mask],
            y_pred[high_conf_mask]
        )
        print(f"\nHigh Confidence (>60%) Accuracy: {high_conf_accuracy:.4f}")
        print(f"High Confidence Predictions: {high_conf_mask.sum()} / {len(y_test)}")

    # Warn if accuracy is near random
    if accuracy < 0.55:
        print("\n[WARNING] Accuracy is close to random guessing. Model likely has no predictive value for this asset.")

    return model, feature_cols, X_test, y_test, y_pred, y_pred_proba, df.iloc[split_idx:]

In [6]:
# ============================================
# 4. BACKTESTING (BINARY)
# ============================================

def backtest_strategy_binary(y_test, y_pred, y_pred_proba, df_test, confidence_threshold=0.6):
    """
    Backtest the prediction strategy (binary)
    """
    returns = df_test['future_return'].values[-len(y_test):]

    # Strategy 1: Trade all predictions (long if Up, flat otherwise)
    strategy_returns_all = y_pred * returns
    cumulative_all = (1 + strategy_returns_all).cumprod()

    # Strategy 2: Only trade high-confidence predictions
    high_conf_mask = y_pred_proba > confidence_threshold
    strategy_returns_conf = np.where(
        high_conf_mask,
        y_pred * returns,
        0
    )
    cumulative_conf = (1 + strategy_returns_conf).cumprod()

    # Buy and hold
    buy_hold_returns = returns
    cumulative_bh = (1 + buy_hold_returns).cumprod()

    print("\n" + "="*50)
    print("BACKTEST RESULTS (BINARY)")
    print("="*50)

    print(f"\nBuy & Hold Return: {(cumulative_bh[-1] - 1) * 100:.2f}%")
    print(f"Strategy (All Predictions) Return: {(cumulative_all[-1] - 1) * 100:.2f}%")
    print(f"Strategy (High Confidence) Return: {(cumulative_conf[-1] - 1) * 100:.2f}%")

    def safe_sharpe(returns):
        std = returns.std()
        return returns.mean() / std * np.sqrt(252) if std > 0 else 0

    sharpe_bh = safe_sharpe(buy_hold_returns)
    sharpe_all = safe_sharpe(strategy_returns_all)
    sharpe_conf = safe_sharpe(strategy_returns_conf)

    print(f"\nSharpe Ratio (Buy & Hold): {sharpe_bh:.2f}")
    print(f"Sharpe Ratio (All): {sharpe_all:.2f}")
    print(f"Sharpe Ratio (High Conf): {sharpe_conf:.2f}")

    print(f"\nNumber of trades (High Conf): {high_conf_mask.sum()}")

    # Warn if strategy underperforms buy-and-hold
    if cumulative_all[-1] < cumulative_bh[-1]:
        print("\n[WARNING] Strategy underperforms buy-and-hold. Model is not profitable on this data.")

    # Print summary
    print("\n=== SUMMARY ===")
    print(f"Test Accuracy: {accuracy_score(y_test, y_pred):.4f}")
    print(f"Buy & Hold Return: {(cumulative_bh[-1] - 1) * 100:.2f}%")
    print(f"Strategy Return: {(cumulative_all[-1] - 1) * 100:.2f}%")
    print(f"Sharpe (Strategy): {safe_sharpe(strategy_returns_all):.2f}")
    if accuracy_score(y_test, y_pred) < 0.55:
        print("NOTE: Model accuracy is not significantly better than random. Consider alternative approaches or more data.")


In [7]:
# ============================================
# 5. USAGE EXAMPLE
# ============================================

# Run full pipeline on aat.us.csv (binary)
print("Training on aat.us.csv (binary)...")

model, features, X_test, y_test, y_pred, y_pred_proba, df_test = train_xgboost_direction_binary(
    data,
    test_size=0.2,
    horizon=1,        # Try different values if needed
    threshold=0.002   # Try different values if needed
)

backtest_strategy_binary(y_test, y_pred, y_pred_proba, df_test, confidence_threshold=0.6)


Training on aat.us.csv (binary)...
Creating features...
Training samples: 760
Test samples: 190
Class distribution (train): [406 354]
[0]	validation_0-logloss:0.67999
[20]	validation_0-logloss:0.67905
[40]	validation_0-logloss:0.67962
[60]	validation_0-logloss:0.68640
[80]	validation_0-logloss:0.69195
[99]	validation_0-logloss:0.69836

EVALUATION RESULTS (BINARY)

Overall Accuracy: 0.4842

Classification Report:
                  precision    recall  f1-score   support

Down/Neutral (0)       0.57      0.57      0.57       114
          Up (1)       0.36      0.36      0.36        76

        accuracy                           0.48       190
       macro avg       0.46      0.46      0.46       190
    weighted avg       0.48      0.48      0.48       190

Confusion Matrix:
[[65 49]
 [49 27]]

Top 10 Most Important Features:
     feature  importance
      rsi_21    0.044522
    bb_width    0.044007
     stoch_d    0.041339
   dist_sma5    0.040169
        macd    0.040075
 bb_position 